# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamr8010/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Go to Colab -> Secrets -> make sure HF_TOKEN exists and is enabled."
    )

print("HF token loaded:", True)

# Load the same dataset used in W05
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

# Use the same 30,000-row working dataset
df = pd.DataFrame(list(ds.take(30000)))

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

HF token loaded: True


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset loaded successfully.
Shape: (30000, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Search and content signals can support prioritization

The research paper presents findings based on search-performance and content-related signals.

**Methodology question:**  
How exactly is the target or outcome defined, and does it represent a measured business outcome or a proxy? I would check whether the label is created from information available after the decision point, because that could affect how the result should be interpreted.

### Finding 2 — Model or rule performance depends on the validation design

The research uses a defined evaluation methodology to assess its findings.

**Methodology question:**  
Does the validation design prevent information from the same client or future periods from appearing in both training and evaluation data? If observations from the same client or overlapping time windows appear on both sides, the measured performance could be more optimistic than performance on genuinely unseen data.

These are constructive methodology questions rather than claims that the findings are incorrect.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
print("Dataset shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content:", df["content_hash_id"].nunique())

Dataset shape: (30000, 30)
Date range: 2025-01-27 to 2025-02-22
Unique clients: 3
Unique content: 5673


In [4]:
# Create a simple observed outcome proxy.
# This is not a business outcome and is used only for validation practice.

df_model = df.copy()

df_model["has_click"] = (
    pd.to_numeric(df_model["gsc_clicks"], errors="coerce")
    .fillna(0)
    .gt(0)
    .astype(int)
)

print("Target distribution:")
print(df_model["has_click"].value_counts())

Target distribution:
has_click
0    27728
1     2272
Name: count, dtype: int64


In [5]:
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_pageviews",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
]

X = df_model[feature_cols].copy()
y = df_model["has_click"].copy()

# Numeric conversion
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Fill missing values using training-safe simple median approach
X = X.fillna(X.median())

print("Features:", feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_pageviews', 'sessions_organic', 'sessions_ai', 'scroll_events']
X shape: (30000, 7)
y shape: (30000,)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_random = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_random.fit(X_train, y_train)

pred_random = model_random.predict(X_test)

random_metrics = {
    "accuracy": accuracy_score(y_test, pred_random),
    "precision": precision_score(y_test, pred_random, zero_division=0),
    "recall": recall_score(y_test, pred_random, zero_division=0),
    "f1": f1_score(y_test, pred_random, zero_division=0)
}

print("Random split results:")
for k, v in random_metrics.items():
    print(f"{k}: {v:.4f}")

Random split results:
accuracy: 0.8583
precision: 0.1678
recall: 0.2203
f1: 0.1905


In [8]:
from sklearn.model_selection import GroupShuffleSplit

groups = df_model["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

model_grouped = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_grouped.fit(X_train_group, y_train_group)

pred_grouped = model_grouped.predict(X_test_group)

grouped_metrics = {
    "accuracy": accuracy_score(y_test_group, pred_grouped),
    "precision": precision_score(y_test_group, pred_grouped, zero_division=0),
    "recall": recall_score(y_test_group, pred_grouped, zero_division=0),
    "f1": f1_score(y_test_group, pred_grouped, zero_division=0)
}

print("Grouped-by-client results:")
for k, v in grouped_metrics.items():
    print(f"{k}: {v:.4f}")

Grouped-by-client results:
accuracy: 0.8912
precision: 0.1622
recall: 0.1797
f1: 0.1705


In [9]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Random split": [
        random_metrics["accuracy"],
        random_metrics["precision"],
        random_metrics["recall"],
        random_metrics["f1"]
    ],
    "Grouped by client": [
        grouped_metrics["accuracy"],
        grouped_metrics["precision"],
        grouped_metrics["recall"],
        grouped_metrics["f1"]
    ]
})

display(comparison)

,Metric,Random split,Grouped by client
0,Accuracy,0.858333,0.891216
1,Precision,0.167785,0.162242
2,Recall,0.220264,0.179739
3,F1,0.190476,0.170543


### Interpretation

The random split provides a less strict estimate because records from the same clients can appear in both training and test sets.

The grouped-by-client split is a more conservative validation design because complete clients are kept on one side of the split.

Therefore, the grouped result is more informative for assessing how the model may behave on unseen clients.

The comparison is treated as measured validation evidence, not as proof of production performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I reviewed the model inputs for information that would not be available at the intended decision moment.

The model uses observed search, analytics, and engagement fields from the current observation.

I do not use future-window labels, future performance fields, or a product-generated action flag as model inputs.

The target `has_click` is treated as an observed proxy for this validation exercise and should not be interpreted as a causal business outcome.

In [10]:
# Explicitly list the model inputs
print("Model features:")
for col in feature_cols:
    print("-", col)

print("\nTarget:")
print("has_click")

# Check that the target is not included in the feature matrix
print("\nTarget leakage into X:", "has_click" in X.columns)

# Check for obvious future/label-like columns by name
suspicious_terms = [
    "label",
    "target",
    "future",
    "outcome",
    "flag",
    "action"
]

suspicious_features = [
    col for col in X.columns
    if any(term in col.lower() for term in suspicious_terms)
]

print("Suspicious feature names:", suspicious_features)

Model features:
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_pageviews
- sessions_organic
- sessions_ai
- scroll_events

Target:
has_click

Target leakage into X: False
Suspicious feature names: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
# Build a small table of grouped-test predictions
error_df = df_model.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions"
    ]
].copy()

error_df["actual"] = y_test_group.values
error_df["predicted"] = pred_grouped

# Keep only incorrect predictions
errors = error_df[
    error_df["actual"] != error_df["predicted"]
].copy()

print("Number of errors:", len(errors))

display(
    errors.head(10)
)

Number of errors: 1070


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,actual,predicted
4009,client_73cda7b4e4f265ea,content_daa3bab7e626fcae,2025-02-11,3,1,25.666667,0,1,0
4010,client_73cda7b4e4f265ea,content_c8a344d6fbdbb873,2025-02-11,4,2,11.500000,0,1,0
4075,client_73cda7b4e4f265ea,content_32edd3b2172aa114,2025-02-11,2,0,3.500000,0,0,1
4083,client_73cda7b4e4f265ea,content_283d482a90ad0d24,2025-02-11,2,0,3.500000,0,0,1
4097,client_73cda7b4e4f265ea,content_722bb352669f7cc0,2025-02-11,5,0,6.000000,0,0,1
4101,client_73cda7b4e4f265ea,content_e032cc1f6c7fef57,2025-02-11,34,0,5.264706,0,0,1
4102,client_73cda7b4e4f265ea,content_787621015327122c,2025-02-11,3,0,5.666667,0,0,1
4105,client_73cda7b4e4f265ea,content_12a5d2a94a3aacb6,2025-02-11,2,0,3.500000,0,0,1
4106,client_73cda7b4e4f265ea,content_69713d44fe697b57,2025-02-11,3,0,5.666667,0,0,1
4112,client_73cda7b4e4f265ea,content_3e6d09468a866dc4,2025-02-11,3,0,1.666667,0,0,1


### Error review

The model's errors show that the available signals do not perfectly separate clicked and non-clicked observations.

A false positive means the model predicted a click but the observed row had no click.

A false negative means the model predicted no click while the observed row had at least one click.

Possible reasons include differences in search demand, ranking position, content quality, and other factors that are not represented by the selected features.

These examples are useful for understanding model limitations, not for claiming a specific causal reason for each error.

## 5. Claim rewrite

### Before

"The model can reliably predict content performance."

### After

"The model measured performance on the selected proxy outcome under the evaluated validation designs. The grouped-by-client result provides a more conservative estimate for unseen clients."

### Before

"The most important features cause better content performance."

### After

"The selected features showed predictive association with the measured proxy in this experiment. The analysis does not establish causality."

### Final safe claim

"On the available anonymized dataset, the model produced measurable predictive performance for the selected proxy outcome. Grouped validation was used to provide a more conservative assessment across clients. The results should be treated as directional decision-support evidence rather than proof of production or causal performance."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.